In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.ndimage import uniform_filter
from scipy.sparse.linalg import spsolve
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

DATA_SEED = 42
SPLIT_SEED = 42
GRID_ROWS = 100
GRID_COLUMNS = 100
N_SAMPLES = GRID_ROWS * GRID_COLUMNS
TEST_FRACTION = 0.20

SPATIAL_RHO = 0.5
N_NEIGHBORS = 8
SPATIAL_SMOOTHING_WINDOW = 5
COORDINATE_JITTER_FRACTION = 0.20
SPATIAL_BLOCK_BINS_PER_AXIS = 10

ROOT_THRESHOLD = 0.20
LEFT_THRESHOLD = -0.10
RIGHT_THRESHOLD = 0.10

BASE_INTERCEPT = 0.0
STRATUM_INTERCEPT_STEP = 0.60
LOCAL_FUNCTION_SCALE = 1.0
ACTIVE_COVARIATE_COUNTS = {
    "S1": 1,
    "S2": 2,
    "S3": 3,
    "S4": 4,
}

NOISE_RATIO = 0.10

OUTPUT_FILENAME = "simulation_data.csv"

TRUE_STRATA = ("S1", "S2", "S3", "S4")
MODEL_FEATURES = [f"x{i}" for i in range(1, 10)]
SPATIAL_METADATA = ["coord_x", "coord_y"]

assert GRID_ROWS >= 2 and GRID_COLUMNS >= 2
assert 1 <= N_NEIGHBORS < N_SAMPLES
assert 0.0 <= SPATIAL_RHO < 1.0
assert 1 <= SPATIAL_SMOOTHING_WINDOW <= min(GRID_ROWS, GRID_COLUMNS)
assert 0.0 <= COORDINATE_JITTER_FRACTION < 0.5
assert 0.0 < TEST_FRACTION < 1.0
assert NOISE_RATIO >= 0.0
assert STRATUM_INTERCEPT_STEP > 0.0
assert LOCAL_FUNCTION_SCALE > 0.0
assert set(ACTIVE_COVARIATE_COUNTS) == set(TRUE_STRATA)
assert list(ACTIVE_COVARIATE_COUNTS.values()) == [1, 2, 3, 4]

seed_sequence = np.random.SeedSequence(DATA_SEED)
rng_location, rng_covariates, rng_noise = [
    np.random.default_rng(seed) for seed in seed_sequence.spawn(3)
]

pd.set_option("display.max_columns", 30)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})


In [ ]:
grid_x = (np.arange(GRID_COLUMNS) + 0.5) * 2.0 / GRID_COLUMNS - 1.0
grid_y = (np.arange(GRID_ROWS) + 0.5) * 2.0 / GRID_ROWS - 1.0
grid_x_mesh, grid_y_mesh = np.meshgrid(grid_x, grid_y)
base_coordinates = np.column_stack([grid_x_mesh.ravel(), grid_y_mesh.ravel()])

grid_spacing_x = 2.0 / GRID_COLUMNS
grid_spacing_y = 2.0 / GRID_ROWS
coord_x = base_coordinates[:, 0] + rng_location.uniform(
    -COORDINATE_JITTER_FRACTION, COORDINATE_JITTER_FRACTION, N_SAMPLES
) * grid_spacing_x
coord_y = base_coordinates[:, 1] + rng_location.uniform(
    -COORDINATE_JITTER_FRACTION, COORDINATE_JITTER_FRACTION, N_SAMPLES
) * grid_spacing_y

neighbor_model = NearestNeighbors(n_neighbors=N_NEIGHBORS + 1)
neighbor_model.fit(base_coordinates)
neighbor_distances, neighbor_indices = neighbor_model.kneighbors(base_coordinates)
neighbor_distances = neighbor_distances[:, 1:]
neighbor_indices = neighbor_indices[:, 1:]

weight_rows = np.repeat(np.arange(N_SAMPLES), N_NEIGHBORS)
weight_columns = neighbor_indices.ravel()
inverse_distance_weights = 1.0 / np.maximum(
    neighbor_distances.ravel(), np.finfo(float).eps
)
spatial_weights = sparse.csr_matrix(
    (inverse_distance_weights, (weight_rows, weight_columns)),
    shape=(N_SAMPLES, N_SAMPLES),
)
row_sums = np.asarray(spatial_weights.sum(axis=1)).ravel()
spatial_weights = sparse.diags(1.0 / row_sums) @ spatial_weights

def generate_spatial_lag_field(innovation):
    system = sparse.eye(N_SAMPLES, format="csr") - SPATIAL_RHO * spatial_weights
    field = np.asarray(spsolve(system, innovation), dtype=float)
    field = uniform_filter(
        field.reshape(GRID_ROWS, GRID_COLUMNS),
        size=SPATIAL_SMOOTHING_WINDOW,
        mode="reflect",
    ).ravel()
    return (field - field.mean()) / field.std(ddof=0)

def morans_i(values):
    centered = np.asarray(values, dtype=float) - np.mean(values)
    numerator = float(centered @ (spatial_weights @ centered))
    denominator = float(centered @ centered)
    return (N_SAMPLES / float(spatial_weights.sum())) * numerator / denominator

X = pd.DataFrame({
    f"x{i}": rng_covariates.normal(0.0, 1.0, N_SAMPLES)
    for i in range(1, 8)
})
X["x8"] = generate_spatial_lag_field(
    rng_covariates.normal(0.0, 1.0, N_SAMPLES)
)
X["x9"] = generate_spatial_lag_field(
    rng_covariates.normal(0.0, 1.0, N_SAMPLES)
)

for column in X.columns:
    X[column] = (X[column] - X[column].mean()) / X[column].std(ddof=0)

spatial_environment_summary = pd.DataFrame([
    {
        "variable": variable,
        "spatial_rho": SPATIAL_RHO,
        "morans_i": morans_i(X[variable].to_numpy()),
        "correlation_with_spatial_lag": np.corrcoef(
            X[variable].to_numpy(), spatial_weights @ X[variable].to_numpy()
        )[0, 1],
    }
    for variable in ["x8", "x9"]
])

display(X.head())
display(spatial_environment_summary.round(4))


In [ ]:
left_root = X["x8"].to_numpy() <= ROOT_THRESHOLD
x9_values = X["x9"].to_numpy()

true_stratum = np.select(
    [
        left_root & (x9_values <= LEFT_THRESHOLD),
        left_root & (x9_values > LEFT_THRESHOLD),
        (~left_root) & (x9_values <= RIGHT_THRESHOLD),
        (~left_root) & (x9_values > RIGHT_THRESHOLD),
    ],
    TRUE_STRATA,
    default="unassigned",
)
assert not np.any(true_stratum == "unassigned")

stratum_counts = (
    pd.Series(true_stratum, name="true_stratum")
    .value_counts()
    .reindex(TRUE_STRATA)
)
display(stratum_counts.to_frame("n"))


In [ ]:
function_rows = []
y_clean = np.empty(N_SAMPLES, dtype=float)

for stratum_index, stratum in enumerate(TRUE_STRATA):
    active_count = ACTIVE_COVARIATE_COUNTS[stratum]
    active_covariates = [f"x{i}" for i in range(1, active_count + 1)]
    intercept = BASE_INTERCEPT + stratum_index * STRATUM_INTERCEPT_STEP
    coefficient = LOCAL_FUNCTION_SCALE / np.sqrt(active_count)
    mask = true_stratum == stratum

    y_clean[mask] = (
        intercept
        + coefficient
        * X.loc[mask, active_covariates].sum(axis=1).to_numpy()
    )
    formula_terms = " + ".join(active_covariates)
    if active_count == 1:
        formula = f"{intercept:.1f} + {formula_terms}"
    else:
        formula = (
            f"{intercept:.1f} + ({formula_terms}) / sqrt({active_count})"
        )
    function_rows.append({
        "true_stratum": stratum,
        "intercept": intercept,
        "active_covariates": ", ".join(active_covariates),
        "active_count": active_count,
        "coefficient_per_covariate": coefficient,
        "function_structure": "normalized linear additive",
        "formula": formula,
    })

function_info = pd.DataFrame(function_rows)
display(function_info)

pooled_within_variance = sum(
    np.mean(true_stratum == stratum)
    * np.var(y_clean[true_stratum == stratum], ddof=0)
    for stratum in TRUE_STRATA
)
within_signal_sd = np.sqrt(pooled_within_variance)
noise_sd = NOISE_RATIO * within_signal_sd
noise = rng_noise.normal(0.0, noise_sd, N_SAMPLES)
y = y_clean + noise

print(f"Pooled within-stratum signal SD: {within_signal_sd:.4f}")
print(f"Noise SD: {noise_sd:.4f} ({NOISE_RATIO:.0%} of within-stratum signal SD)")
print(f"Global clean-response SD: {np.std(y_clean, ddof=0):.4f}")


In [ ]:
all_indices = np.arange(N_SAMPLES)
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_FRACTION,
    random_state=SPLIT_SEED,
    shuffle=True,
    stratify=true_stratum,
)

dataset_split = np.full(N_SAMPLES, "train", dtype=object)
dataset_split[test_indices] = "test"

data = pd.DataFrame({
    "sample_id": [f"SIM_{i:05d}" for i in range(1, N_SAMPLES + 1)],
    **{column: X[column].to_numpy() for column in X.columns},
    "coord_x": coord_x,
    "coord_y": coord_y,
    "y": y,
    "y_clean": y_clean,
    "noise": noise,
    "true_stratum": true_stratum,
    "dataset_split": dataset_split,
})

export_columns = (
    ["sample_id"]
    + MODEL_FEATURES
    + SPATIAL_METADATA
    + ["y", "y_clean", "noise", "true_stratum", "dataset_split"]
)
export_data = data[export_columns].copy()
display(export_data.head())


In [ ]:
def geodetector_q(values, labels):
    values = np.asarray(values, dtype=float).ravel()
    labels = np.asarray(labels).ravel()
    total = values.size * np.var(values, ddof=0)
    if total <= np.finfo(float).eps:
        return 0.0
    within = 0.0
    for label in np.unique(labels):
        selected = values[labels == label]
        within += selected.size * np.var(selected, ddof=0)
    return float(np.clip(1.0 - within / total, 0.0, 1.0))

assert len(export_data) == N_SAMPLES
assert not export_data.isna().any().any()
assert np.isfinite(export_data[MODEL_FEATURES + ["y", "y_clean", "noise"]].to_numpy()).all()
assert MODEL_FEATURES == [f"x{i}" for i in range(1, 10)]
assert not set(SPATIAL_METADATA).intersection(MODEL_FEATURES)
assert export_data["coord_x"].between(-1.0, 1.0).all()
assert export_data["coord_y"].between(-1.0, 1.0).all()
assert np.allclose(np.asarray(spatial_weights.sum(axis=1)).ravel(), 1.0)
assert set(export_data["dataset_split"].unique()) == {"train", "test"}
assert (export_data["dataset_split"] == "train").sum() == len(train_indices)
assert (export_data["dataset_split"] == "test").sum() == len(test_indices)
assert set(export_data["true_stratum"].unique()) == set(TRUE_STRATA)

split_stratum_counts = pd.crosstab(
    export_data["true_stratum"], export_data["dataset_split"]
).reindex(index=TRUE_STRATA, columns=["train", "test"])
assert (split_stratum_counts > 0).all().all()

coordinates = export_data[["coord_x", "coord_y"]].to_numpy(dtype=float)
block_bins = np.floor(
    (coordinates + 1.0) * SPATIAL_BLOCK_BINS_PER_AXIS / 2.0
).astype(int)
block_bins = np.clip(
    block_bins, 0, SPATIAL_BLOCK_BINS_PER_AXIS - 1
)
diagnostic_block = (
    block_bins[:, 1] * SPATIAL_BLOCK_BINS_PER_AXIS + block_bins[:, 0] + 1
)
block_stratum_counts = pd.crosstab(diagnostic_block, true_stratum)
block_sizes = block_stratum_counts.sum(axis=1)
block_purity = block_stratum_counts.max(axis=1) / block_sizes
block_summary = pd.DataFrame({
    "metric": [
        "n_blocks", "mean_samples_per_block", "minimum_samples_per_block",
        "maximum_samples_per_block", "mean_stratum_purity",
        "median_stratum_purity", "fraction_blocks_purity_ge_0.8",
    ],
    "value": [
        len(block_sizes), block_sizes.mean(), block_sizes.min(), block_sizes.max(),
        block_purity.mean(), block_purity.median(), (block_purity >= 0.8).mean(),
    ],
})

stratum_summary = (
    data.groupby("true_stratum", sort=True)
    .agg(
        n=("sample_id", "size"),
        y_clean_mean=("y_clean", "mean"),
        y_clean_sd=("y_clean", "std"),
        y_mean=("y", "mean"),
        y_sd=("y", "std"),
    )
)

true_s1_path = left_root & (x9_values <= LEFT_THRESHOLD)
left_child_labels = (x9_values[left_root] <= LEFT_THRESHOLD).astype(int)
right_child_labels = (x9_values[~left_root] <= RIGHT_THRESHOLD).astype(int)
q_summary = pd.DataFrame({
    "partition": [
        "true four strata",
        "true root",
        "left child conditional split",
        "right child conditional split",
        "true S1 compound path",
    ],
    "q_y": [
        geodetector_q(y, true_stratum),
        geodetector_q(y, left_root.astype(int)),
        geodetector_q(y[left_root], left_child_labels),
        geodetector_q(y[~left_root], right_child_labels),
        geodetector_q(y, true_s1_path.astype(int)),
    ],
})

display(stratum_summary.round(4))
display(split_stratum_counts)
display(spatial_environment_summary.round(4))
display(block_summary.round(4))
display(q_summary.round(4))
print(f"Rows: {len(export_data):,}")
print(f"Train rows: {(export_data['dataset_split'] == 'train').sum():,}")
print(f"Test rows: {(export_data['dataset_split'] == 'test').sum():,}")
print("All validation checks passed.")


In [ ]:
stratum_order = list(TRUE_STRATA)
stratum_colors = {
    "S1": "#4C78A8",
    "S2": "#F58518",
    "S3": "#54A24B",
    "S4": "#E45756",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

for stratum in stratum_order:
    subset = data[data["true_stratum"] == stratum]
    axes[0, 0].scatter(
        subset["coord_x"], subset["coord_y"],
        s=9, alpha=0.65, color=stratum_colors[stratum],
        label=stratum, linewidths=0,
    )
axes[0, 0].set(
    title="Spatial pattern (coordinates are metadata only)",
    xlabel="coord_x", ylabel="coord_y",
    xlim=(-1, 1), ylim=(-1, 1), aspect="equal",
)
axes[0, 0].legend(frameon=False, markerscale=2)

for stratum in stratum_order:
    subset = data[data["true_stratum"] == stratum]
    axes[0, 1].scatter(
        subset["x8"], subset["x9"],
        s=9, alpha=0.65, color=stratum_colors[stratum],
        label=stratum, linewidths=0,
    )
axes[0, 1].axvline(ROOT_THRESHOLD, color="black", linestyle="--", linewidth=1.2)
axes[0, 1].plot(
    [data["x8"].min(), ROOT_THRESHOLD],
    [LEFT_THRESHOLD, LEFT_THRESHOLD],
    color="black", linestyle="--", linewidth=1.2,
)
axes[0, 1].plot(
    [ROOT_THRESHOLD, data["x8"].max()],
    [RIGHT_THRESHOLD, RIGHT_THRESHOLD],
    color="black", linestyle="--", linewidth=1.2,
)
axes[0, 1].set(
    title="Known compound rules in environmental space",
    xlabel="x8", ylabel="x9",
)

for axis, variable in zip(axes[1], ["x8", "x9"]):
    field_plot = axis.scatter(
        data["coord_x"], data["coord_y"], c=data[variable],
        cmap="coolwarm", s=11, alpha=0.9, linewidths=0,
    )
    axis.set(
        title=f"Spatial-lag environmental field: {variable}",
        xlabel="coord_x", ylabel="coord_y",
        xlim=(-1, 1), ylim=(-1, 1), aspect="equal",
    )
    fig.colorbar(field_plot, ax=axis, label=variable)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

response_plot = axes[0].scatter(
    data["coord_x"], data["coord_y"], c=data["y"],
    cmap="viridis", s=10, alpha=0.80, linewidths=0,
)
axes[0].set(
    title="Noisy simulated response in space",
    xlabel="coord_x", ylabel="coord_y",
    xlim=(-1, 1), ylim=(-1, 1), aspect="equal",
)
fig.colorbar(response_plot, ax=axes[0], label="y")

split_styles = {
    "train": {"color": "#4C78A8", "marker": "o", "alpha": 0.35},
    "test": {"color": "#E45756", "marker": "x", "alpha": 0.75},
}
for split_name in ["train", "test"]:
    subset = data[data["dataset_split"] == split_name]
    axes[1].scatter(
        subset["coord_x"], subset["coord_y"],
        s=12, linewidths=0.5, label=split_name,
        **split_styles[split_name],
    )
axes[1].set(
    title="Fixed stratified random split (not spatial CV)",
    xlabel="coord_x", ylabel="coord_y",
    xlim=(-1, 1), ylim=(-1, 1), aspect="equal",
)
axes[1].legend(frameon=False)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for axis, stratum in zip(axes.flat, stratum_order):
    subset = data[data["true_stratum"] == stratum]
    axis.scatter(
        subset["y_clean"], subset["y"],
        s=10, alpha=0.55, color=stratum_colors[stratum], linewidths=0,
    )
    lower = min(subset["y_clean"].min(), subset["y"].min())
    upper = max(subset["y_clean"].max(), subset["y"].max())
    axis.plot([lower, upper], [lower, upper], color="black", linestyle="--", linewidth=1)
    active_count = int(function_info.set_index("true_stratum").loc[stratum, "active_count"])
    axis.set(
        title=f"{stratum}: {active_count} active covariates",
        xlabel="Clean response", ylabel="Noisy response",
    )
fig.suptitle("Noise around each stratum-specific function", fontsize=13)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

mean_table = stratum_summary.loc[stratum_order, ["y_clean_mean", "y_mean"]]
mean_table.plot(kind="bar", ax=axes[0], color=["#72B7B2", "#4C78A8"])
axes[0].set(
    title="Moderate response-level differences by stratum",
    xlabel="true stratum", ylabel="mean response",
)
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(["clean mean", "noisy mean"], frameon=False)

count_matrix = split_stratum_counts.loc[stratum_order, ["train", "test"]].to_numpy()
heatmap = axes[1].imshow(count_matrix, cmap="Blues", aspect="auto")
axes[1].set(
    title="Sample count by true stratum and fixed split",
    xlabel="dataset split", ylabel="true stratum",
    xticks=np.arange(2), yticks=np.arange(4),
    xticklabels=["train", "test"], yticklabels=stratum_order,
)
for row in range(count_matrix.shape[0]):
    for column in range(count_matrix.shape[1]):
        axes[1].text(column, row, count_matrix[row, column], ha="center", va="center")
fig.colorbar(heatmap, ax=axes[1], label="sample count")
plt.show()


In [ ]:
OUTPUT_CSV = Path(OUTPUT_FILENAME)
export_data.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"CSV written to: {OUTPUT_CSV.resolve()}")
